# Notebook 5 v2: new primary model and all remaining MIMIC-IV analyses (aortic aneurysm AKI)
**Fresh Colab session. Run the cells in order (Runtime > Run all works).** If Cell 0 says FIX THESE FIRST, edit only the paths in its CONFIG block.

Everything prints in aggregate form. Counts below 11 print as `<11` (change SMALL_CELL in Cell 0 if your DUA or your institution sets a different minimum). The last cell (Cell 6) repeats everything printed by the earlier cells.

Checks printed at the top of Cell 0 must read: baseline available 739, recomputed AKI 229, disagreements 0, and the old primary OR 2.01 (1.35-2.97), N 740, 229 events. If not, stop and check the earlier output.

v2.2 (29 Oct 2026) adds Cell 5 (Table 1 by approach, unadjusted RR and RD, location interaction, in-hospital mortality, S2b with verified vasopressors only, and diagnostics for why the ICU-arrival analysis R3 differs) and complementary suppression so a hidden small cell cannot be recovered by subtraction from a known total. The final print is now Cell 6.

Changes from v1: small-cell masking added to the missingness table, raw admission-type counts and the CHECK lines; Cell 0 no longer fails if your cohort file already carries admittime/dischtime; S4c relabelled; Cell 4 no longer calls overlap weighting doubly robust; Cell 1 stops with a clear message if the primary model fails.

**v2.2 adds Cell 7 (vasopressor definition check) and Cell 8 (MIMIC thoracic-vs-abdominal model for the eICU comparison).**

**v2.3 (2 Nov 2026) adds Cell 9: dialysis (renal replacement therapy) during the stay. Run all cells first (it needs their variables).** It needs two files from the MIMIC-IV icu folder: d_items and procedureevents. If it stops with a missing-file message, edit DI_CAND / PE_CAND at the top of Cell 9.

In [ ]:
# Cell 0: setup, loading, preflight (EDIT ONLY THE PATHS IF A FILE IS REPORTED MISSING)
import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass

# ---- CONFIG ----
P = "/content/drive/MyDrive/"
F = dict(cohort = P+"mimic-iv/aortic_aneurysm_final_modeling_dataset_v4.csv",
         adm    = P+"mimic-iv/hosp/admissions.csv.gz",
         cr     = P+"mimic-iv/aortic_aneurysm_creatinine_raw.csv",
         proc   = P+"mimic-iv/hosp/procedures_icd.csv.gz",
         dx     = P+"mimic-iv/hosp/diagnoses_icd.csv.gz",
         inp    = P+"MIMIC 4/inputevents.csv.gz")
ICU_CANDIDATES = [P+"mimic-iv/icu/icustays.csv.gz", P+"MIMIC 4/icustays.csv.gz", P+"mimic-iv/icustays.csv.gz"]   # optional (analysis R3)
CR_ITEM = 50912
VASO_ITEMS = [221289,221653,221662,221749,221906,222315,229617,229630,229631,229632]
PROC9  = ('3844','3971','3973','3978'); PROC10 = ('04R','04V','02RW','02VW')
SMALL_CELL = 11          # counts below this are printed as '<11'. Change to your DUA / institutionalal minimum if different.
N_BOOT = 1000            # clustered bootstrap repetitions
# ----------------

REPORT = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)
def n_(v):
    v = int(v); return '<%d' % SMALL_CELL if 0 < v < SMALL_CELL else str(v)
def cnt(x):
    """mask small counts in a count table (rows = groups with KNOWN totals). If a row would have exactly one masked cell,
    also suppress that row's smallest other cell ('*'), so the hidden cell cannot be recovered by subtraction."""
    x = pd.DataFrame(x); out = x.astype(object).copy()
    for i in range(x.shape[0]):
        row = list(x.iloc[i]); small = [j for j, v in enumerate(row) if isinstance(v, (int, np.integer)) and 0 < v < SMALL_CELL]
        sec = []
        if len(small) == 1:
            oth = [(v, j) for j, v in enumerate(row) if j not in small and isinstance(v, (int, np.integer)) and v > 0]
            if oth: sec = [min(oth)[1]]
        for j in range(x.shape[1]):
            out.iat[i, j] = ('<%d' % SMALL_CELL) if j in small else ('*' if j in sec else row[j])
    return out
def mask_dict(s):
    """mask small counts in a value_counts() result"""
    return {k: n_(v) for k, v in s.items()}

REQ = dict(cohort=['hadm_id','aki','approach','location','age_at_admission','gender','cci','anchor_year_group','rupture','vasopressor'],
           adm=['subject_id','hadm_id','admittime','dischtime','admission_type','hospital_expire_flag'],
           cr=['hadm_id','itemid','charttime','valuenum'], proc=['hadm_id','chartdate','icd_code','icd_version'],
           dx=['hadm_id','icd_code','icd_version'], inp=['hadm_id','starttime','itemid'])
problems = []
for k, cols in REQ.items():
    if not os.path.exists(F[k]): problems.append(f"MISSING FILE for '{k}': {F[k]}"); continue
    head = pd.read_csv(F[k], nrows=5); miss = [c for c in cols if c not in head.columns]
    if miss: problems.append(f"'{k}' file lacks columns {miss}; it has {list(head.columns)}")
if problems: raise SystemExit("FIX THESE FIRST:\n  " + "\n  ".join(problems))

COHORT = pd.read_csv(F['cohort']); ids = set(COHORT.hadm_id)
assert set(COHORT.approach.dropna().unique()) <= {'open','endovascular'}
assert 'abdominal' in set(COHORT.location.dropna().unique())
ADM = pd.read_csv(F['adm'], usecols=['subject_id','hadm_id','admittime','dischtime','admission_type','hospital_expire_flag'], parse_dates=['admittime','dischtime'])
ADM = ADM[ADM.hadm_id.isin(ids)]
# the MIMIC admissions table is authoritative for these columns: if your cohort file already carries any, drop THEIR copy (not ADM's)
clash = [c_ for c_ in ADM.columns if c_ in COHORT.columns and c_ != 'hadm_id']
if clash: say("NOTE: cohort file also carried", clash, "- using the MIMIC admissions table versions")
COHORT = COHORT.drop(columns=clash)
LABS = pd.read_csv(F['cr'], parse_dates=['charttime'])
LABS = LABS.drop(columns=[c for c in ['admittime','dischtime'] if c in LABS.columns])
# creatinine: everything up to discharge, INCLUDING values charted before the admission timestamp (e.g., ED draws)
cr = (LABS[(LABS.itemid==CR_ITEM) & LABS.hadm_id.isin(ids)][['hadm_id','charttime','valuenum']].dropna()
      .merge(ADM[['hadm_id','admittime','dischtime']], on='hadm_id'))
cr = cr[cr.charttime <= cr.dischtime].sort_values(['hadm_id','charttime'])

pr = pd.read_csv(F['proc'], dtype={'icd_code':str}, parse_dates=['chartdate']); pr = pr[pr.hadm_id.isin(ids)]
m9 = (pr.icd_version==9) & pr.icd_code.str.startswith(PROC9); m10 = (pr.icd_version==10) & pr.icd_code.str.startswith(PROC10)
proc_date = pr[m9|m10].groupby('hadm_id').chartdate.min().rename('proc_date')
has3978 = set(pr[(pr.icd_version==9) & pr.icd_code.str.startswith('3978')].hadm_id)

vas = []
for ch in pd.read_csv(F['inp'], usecols=['hadm_id','starttime','itemid'], parse_dates=['starttime'], chunksize=2_000_000):
    vas.append(ch[ch.itemid.isin(VASO_ITEMS) & ch.hadm_id.isin(ids)])
vaso_first = pd.concat(vas).groupby('hadm_id').starttime.min().rename('vaso_first')

d = COHORT.merge(ADM, on='hadm_id', how='left').merge(proc_date, on='hadm_id', how='left').merge(vaso_first, on='hadm_id', how='left')
def urg(x):
    x = str(x).upper()
    if 'ELECTIVE' in x or 'SURGICAL SAME DAY' in x: return 'elective'
    if 'OBSERV' in x: return 'observation'
    return 'emergency_urgent'
d['urg'] = d.admission_type.map(urg)
say("Software: python pandas", pd.__version__, "| statsmodels", sm.__version__, "| run", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))
say("PREFLIGHT OK | cohort rows:", len(d), "| patients:", d.subject_id.nunique(), "| AKI events:", n_(d.aki.sum()))
say("Raw admission types (small counts masked):", mask_dict(d.admission_type.value_counts()))
say("Urgency groups used (small counts masked):", mask_dict(d.urg.value_counts()), " (manuscript Table 1 shows elective 342 [46.2%]; check if elective differs)")

# ---- model helpers ----
RESULTS = []
def fit(data, y, terms, label, family='poisson', quiet=False):
    try: return _fit(data, y, terms, label, family, quiet)
    except Exception as e:
        say(f"{label}: MODEL FAILED ({type(e).__name__}: {str(e)[:70]}). Probably too few events or a constant covariate; check this line.")
        return None
def _fit(data, y, terms, label, family='poisson', quiet=False):
    f = f"{y} ~ " + " + ".join(terms)
    fam = sm.families.Poisson() if family=='poisson' else sm.families.Binomial()
    m0 = smf.glm(f, data=data, family=fam).fit()
    g = data.loc[m0.model.data.row_labels, 'subject_id'].values
    m = smf.glm(f, data=data, family=fam).fit(cov_type='cluster', cov_kwds={'groups': g})
    t = [k for k in m.params.index if 'approach' in k][0]
    lo, hi = np.exp(m.conf_int().loc[t]); ev = int(data.loc[m.model.data.row_labels, y].sum())
    row = dict(model=label, N=int(m.nobs), events=ev, est=float(np.exp(m.params[t])), lo=float(lo), hi=float(hi), p=float(m.pvalues[t]), measure='RR' if family=='poisson' else 'OR')
    RESULTS.append(row)
    if not quiet: say(f"{label}: N={row['N']}, events={n_(ev)}, open vs endovascular {row['measure']} {row['est']:.2f} ({lo:.2f} to {hi:.2f}), p={row['p']:.3f}")
    return m
PRE = ["C(approach, Treatment('endovascular'))","C(location, Treatment('abdominal'))","age_at_admission","C(gender)","cci","C(anchor_year_group)","rupture","C(urg, Treatment('elective'))"]
def T(extra=()): return PRE + list(extra)
COV_FORMULA = "C(location, Treatment('abdominal')) + age_at_admission + C(gender) + cci + C(anchor_year_group) + rupture + C(urg, Treatment('elective'))"

# ---- creatinine summaries ----
c = cr.merge(d[['hadm_id','proc_date']], on='hadm_id')
f24 = c[c.charttime <= c.admittime + pd.Timedelta('24h')]
d = d.merge(f24.groupby('hadm_id').valuenum.min().rename('base24'), on='hadm_id', how='left') \
     .merge(c.groupby('hadm_id').valuenum.max().rename('peak_all'), on='hadm_id', how='left')
d['aki_re'] = ((d.peak_all/d.base24) >= 1.5).astype(int)
say(f"CHECK: baseline available {int(d.base24.notna().sum())} | cohort admissions with NO baseline {n_(d.base24.isna().sum())} | recomputed AKI events {int(d.aki_re.sum())} | manuscript flag events {int(d.aki.sum())} | disagreements {n_((d.aki_re!=d.aki)[d.base24.notna()].sum())}")

# ---- the old primary must reproduce 2.01 (OR) ----
OLD = ["C(approach, Treatment('endovascular'))","C(location, Treatment('abdominal'))","age_at_admission","C(gender)","cci","C(anchor_year_group)","rupture","vasopressor"]
m_old = smf.glm("aki ~ " + " + ".join(OLD), data=d, family=sm.families.Binomial()).fit()
t_ = [k for k in m_old.params.index if 'approach' in k][0]; lo_, hi_ = np.exp(m_old.conf_int().loc[t_])
say(f"CHECK: manuscript primary (reproduce OR 2.01, 1.35-2.97; N 740; events 229): N={int(m_old.nobs)}, OR {np.exp(m_old.params[t_]):.2f} ({lo_:.2f}-{hi_:.2f})")

In [ ]:
# Cell 1: R1 NEW PRIMARY MODEL. Pre-procedure confounders only; modified Poisson risk ratio; cluster-robust by patient
say("\n=== R1. NEW PRIMARY (pre-procedure confounders: approach, location, age, sex, CCI, era, rupture, admission urgency) ===")
m_rr = fit(d, 'aki', T(), 'R1  PRIMARY: modified Poisson risk ratio, cluster by patient', 'poisson')
if m_rr is None: raise SystemExit("The primary model failed; see the MODEL FAILED line above. Nothing below can run without it.")
est, lo, hi = RESULTS[-1]['est'], RESULTS[-1]['lo'], RESULTS[-1]['hi']
m_or = fit(d, 'aki', T(), 'R1b Same covariates, logistic odds ratio (secondary)', 'logit')

def gcomp(model, data):
    a = data.copy(); a['approach'] = 'open';         p1 = model.predict(a).mean()
    b = data.copy(); b['approach'] = 'endovascular'; p0 = model.predict(b).mean()
    return p1, p0
fml = "aki ~ " + " + ".join(T())
mb = smf.glm(fml, data=d, family=sm.families.Binomial()).fit()
p1, p0 = gcomp(mb, d)
groups = d.reset_index(drop=True).groupby('subject_id').indices; keys = np.array(list(groups.keys()))
dd = d.reset_index(drop=True); rng = np.random.default_rng(20261027); bs = []
for _ in range(N_BOOT):
    pick = rng.choice(keys, size=len(keys), replace=True)
    idx = np.concatenate([groups[k] for k in pick]); s = dd.iloc[idx]
    try:
        q1, q0 = gcomp(smf.glm(fml, data=s, family=sm.families.Binomial()).fit(disp=0), s); bs.append((q1-q0, q1/q0))
    except Exception: pass
bs = np.array(bs)
say(f"Standardized (g-computation) risk if all open {p1:.1%} vs all endovascular {p0:.1%}; risk difference {100*(p1-p0):.1f} points "
    f"(95% CI {100*np.percentile(bs[:,0],2.5):.1f} to {100*np.percentile(bs[:,0],97.5):.1f}); standardized RR {p1/p0:.2f} "
    f"({np.percentile(bs[:,1],2.5):.2f} to {np.percentile(bs[:,1],97.5):.2f}); {len(bs)} patient-clustered bootstrap reps, seed 20261027")
RESULTS.append(dict(model='R1c Standardized risk difference (points)', N=len(d), events=int(d.aki.sum()), est=100*(p1-p0), lo=100*np.percentile(bs[:,0],2.5), hi=100*np.percentile(bs[:,0],97.5), p=np.nan, measure='RD'))

def evalue(r):
    r = 1/r if r < 1 else r
    return r + np.sqrt(r*(r-1))
lim = lo if est > 1 else hi
say(f"E-value for the primary RR {est:.2f}: {evalue(est):.2f}; for the CI limit nearest the null ({lim:.2f}): {1.0 if (lo<=1<=hi) else evalue(lim):.2f}")
say("Crude AKI: open", f"{d[d.approach=='open'].aki.mean():.1%}", "| endovascular", f"{d[d.approach=='endovascular'].aki.mean():.1%}", "| overall", f"{d.aki.mean():.1%}")

In [ ]:
# Cell 2: SENSITIVITY analyses on the new primary (all labelled post hoc)
say("\n=== SENSITIVITY ANALYSES (post hoc) ===")
# S1 first admission per patient
first = d.sort_values('admittime').drop_duplicates('subject_id')
fit(first, 'aki', T(), 'S1  First admission per patient only', 'poisson')
# S2 add the (post-procedure) vasopressor, i.e., the old model structure with urgency
fit(d, 'aki', T(['vasopressor']), 'S2  Add any-vasopressor (post-procedure variable, mediator risk)', 'poisson')
# S3 CKD stage and baseline creatinine
dx = pd.read_csv(F['dx'], usecols=['hadm_id','icd_code','icd_version'], dtype={'icd_code':str}); dx = dx[dx.hadm_id.isin(ids)]
def stage(code, v):
    if v==9: return {'5851':1,'5852':2,'5853':3,'5854':4,'5855':5,'5859':9}.get(code[:4])
    for s,k in [('N181',1),('N182',2),('N183',3),('N184',4),('N185',5),('N189',9)]:
        if code.startswith(s): return k
dx['stage'] = [stage(c_,v_) for c_,v_ in zip(dx.icd_code, dx.icd_version)]
st = dx.dropna(subset=['stage']).groupby('hadm_id').stage.apply(set)
def grp(s):
    if not isinstance(s,set): return 'none'
    if s & {4,5}: return 'stage4-5'
    if 3 in s: return 'stage3'
    if s & {1,2}: return 'stage1-2'
    return 'unspecified'
d['ckd_group'] = d.hadm_id.map(st).apply(grp); d['log_base_cr'] = np.log(d.base24)
fit(d, 'aki', T(["C(ckd_group, Treatment('none'))"]), 'S3a Add CKD stage group', 'poisson')
fit(d, 'aki', T(['log_base_cr']), 'S3b Add log(baseline creatinine)', 'poisson')
# S4 exclusions
fit(d[d.rupture==0], 'aki', [t for t in T() if t!='rupture'], 'S4a Excluding ruptured repairs', 'poisson')
fit(d[d.urg=='elective'], 'aki', [t for t in T() if 'urg' not in t], 'S4b Elective admissions only', 'poisson')
fit(d[~d.hadm_id.isin(has3978)], 'aki', T(), 'S4c Excluding any remaining admission with an ICD-9 3978 code (resolves the one mixed 3978+3971 admission; at most one expected)', 'poisson')
say("Admissions in the analysed cohort with an ICD-9 3978 code:", n_(d.hadm_id.isin(has3978).sum()))
# S5 death as a competing event
d['died'] = (d.hospital_expire_flag==1).astype(int); d['aki_or_death'] = ((d.aki==1)|(d.died==1)).astype(int)
t = pd.crosstab(d.approach, [d.died, d.aki]); t.columns = [f"died={a},aki={b}" for a,b in t.columns]
say("In-hospital death by AKI status and approach (counts; small cells masked):"); say(cnt(t).to_string())
fit(d, 'aki_or_death', T(), 'S5  Composite outcome: AKI or in-hospital death', 'poisson')
# S6 detection bias: creatinine draws per day at risk
ncr = cr.groupby('hadm_id').size().rename('n_cr'); d = d.drop(columns=[x for x in ['n_cr'] if x in d.columns]).merge(ncr, on='hadm_id', how='left')
d['days'] = ((d.dischtime-d.admittime).dt.total_seconds()/86400).clip(lower=1); d['cr_per_day'] = d.n_cr/d.days; d['log_cr_per_day'] = np.log(d.cr_per_day.clip(lower=0.05))
say("Creatinine draws per day (median [IQR]) by approach:"); say(d.groupby('approach').cr_per_day.describe()[['25%','50%','75%']].round(2).to_string())
say("... by AKI status:"); say(d.groupby('aki').cr_per_day.describe()[['25%','50%','75%']].round(2).to_string())
say("Length of stay, days (median [IQR]) by approach:"); say(d.groupby('approach').days.describe()[['25%','50%','75%']].round(1).to_string())
fit(d, 'aki', T(['log_cr_per_day']), 'S6  Add log(creatinine draws per day)', 'poisson')

In [ ]:
# Cell 3: OUTCOME DEFINITION sensitivity (KDIGO severity stages, absolute-rise criterion) and BASELINE TIMING
say("\n=== OUTCOME DEFINITIONS ===")
d['ratio'] = d.peak_all/d.base24
d['stage2plus'] = (d.ratio>=2.0).astype(int); d['stage3'] = (d.ratio>=3.0).astype(int)
say("KDIGO creatinine-ratio stages (stage 1 = 1.5-1.99x, 2 = 2.0-2.99x, 3 = >=3.0x). Counts by approach (small cells masked):")
sg = pd.cut(d.ratio, [0,1.5,2.0,3.0,np.inf], right=False, labels=['no AKI','stage 1','stage 2','stage 3'])
say(cnt(pd.crosstab(d.approach, sg)).to_string())
for yv, lab in [('stage2plus','O1  Stage 2 or 3 (ratio >= 2.0)'), ('stage3','O2  Stage 3 (ratio >= 3.0)')]:
    try: fit(d, yv, T(), lab, 'poisson')
    except Exception as e: say(lab, ': model did not converge (too few events):', str(e)[:80])
# absolute rise >=0.3 mg/dL within 48 h
def rise03(g):
    t = g.charttime.values.astype('datetime64[s]').astype('int64'); v = g.valuenum.values
    for i in range(len(v)):
        w = (t[i]-t[:i+1] <= 48*3600)
        if v[i] - v[:i+1][w].min() >= 0.3: return 1
    return 0
r3 = cr.groupby('hadm_id').apply(rise03).rename('aki_03'); d = d.drop(columns=[x for x in ['aki_03'] if x in d.columns]).merge(r3, on='hadm_id', how='left'); d['aki_03'] = d.aki_03.fillna(0).astype(int)
d['aki_any'] = ((d.aki==1)|(d.aki_03==1)).astype(int)
say("Absolute-rise criterion (>=0.3 mg/dL within 48 h): events", n_(d.aki_03.sum()), "| ratio OR rise:", n_(d.aki_any.sum()), "| overlap with the ratio criterion:", n_(((d.aki==1)&(d.aki_03==1)).sum()))
fit(d, 'aki_03', T(), 'O3  Absolute rise >= 0.3 mg/dL within 48 h', 'poisson')
fit(d, 'aki_any', T(), 'O4  Ratio >= 1.5 OR absolute rise >= 0.3 (combined creatinine criteria)', 'poisson')

say("\n=== BASELINE TIMING (is the baseline before the procedure?) ===")
c = cr.merge(d[['hadm_id','proc_date']], on='hadm_id'); f24 = c[c.charttime <= c.admittime + pd.Timedelta('24h')]
base_pre = f24[f24.charttime < f24.proc_date].groupby('hadm_id').valuenum.min().rename('base_pre')
peak_post = c[c.charttime >= c.proc_date].groupby('hadm_id').valuenum.max().rename('peak_post')
d = d.drop(columns=[x for x in ['base_pre','peak_post'] if x in d.columns]).merge(base_pre, on='hadm_id', how='left').merge(peak_post, on='hadm_id', how='left')
nopre = d.base24.notna() & d.base_pre.isna()
say(f"Admissions with a baseline in the first 24 h: {int(d.base24.notna().sum())}; with a creatinine charted before the procedure DAY: {int(d.base_pre.notna().sum())}; "
    f"baseline only on/after the procedure day: {n_(nopre.sum())} ({nopre.mean():.1%}). Procedure date is day-level, so this is an upper bound for post-procedure baselines.")
say("Without a pre-procedure baseline, by approach (share):"); say(d.assign(x=nopre).groupby('approach').x.mean().round(3).to_string())
say("Admission-to-procedure days (25th, 50th, 75th):", ((d.proc_date-d.admittime.dt.normalize()).dt.days).describe()[['25%','50%','75%']].tolist())
d1 = d[d.base_pre.notna()].copy(); d1['aki_K'] = (d1.peak_all/d1.base_pre >= 1.5).astype(int); d1['aki_N'] = ((d1.peak_post/d1.base_pre >= 1.5) & d1.peak_post.notna()).astype(int)
fit(d1, 'aki',   T(), 'K0 Primary definition, only admissions with a pre-procedure baseline', 'poisson')
fit(d1, 'aki_K', T(), 'K  Baseline before the procedure day; peak over the admission', 'poisson')
fit(d1[d1.peak_post.notna()], 'aki_N', T(), 'N  Baseline before the procedure day; peak on/after the procedure day', 'poisson')

icu_path = next((p for p in ICU_CANDIDATES if os.path.exists(p)), None)
if icu_path:
    icu = pd.read_csv(icu_path, usecols=['hadm_id','intime'], parse_dates=['intime']); icu = icu[icu.hadm_id.isin(ids)].groupby('hadm_id').intime.min().rename('icu_in')
    c2 = cr.merge(icu, on='hadm_id'); b_icu = c2[c2.charttime <= np.minimum(c2.icu_in, c2.admittime + pd.Timedelta('24h'))].groupby('hadm_id').valuenum.min().rename('base_icu')
    p_icu = c2[c2.charttime > c2.icu_in].groupby('hadm_id').valuenum.max().rename('peak_icu')
    d = d.drop(columns=[x for x in ['base_icu','peak_icu'] if x in d.columns]).merge(b_icu, on='hadm_id', how='left').merge(p_icu, on='hadm_id', how='left')
    d2 = d[d.base_icu.notna() & d.peak_icu.notna()].copy(); d2['aki_icu'] = (d2.peak_icu/d2.base_icu >= 1.5).astype(int)
    say("R3 ICU-arrival cut: baseline = lowest creatinine before ICU arrival (within the first 24 h); outcome = peak after ICU arrival. Admissions usable:", len(d2))
    fit(d2, 'aki_icu', T(), 'R3a Baseline before first ICU arrival, outcome after it', 'poisson')
    fit(d2[d2.urg=='elective'], 'aki_icu', [t for t in T() if 'urg' not in t], 'R3b ... elective admissions only', 'poisson')
else:
    say("R3 skipped: icustays file not found (optional). Set ICU_CANDIDATES to its path, or ignore.")

In [ ]:
# Cell 4: overlap weighting (a second, weighting-based estimate; NOT doubly robust, and it shares the same unmeasured-confounding limits) and covariate balance (SMD)
import patsy
say("\n=== OVERLAP WEIGHTING AND BALANCE ===")
d['open'] = (d.approach=='open').astype(int)
X = patsy.dmatrix(COV_FORMULA, d, return_type='dataframe'); Xc = X.drop(columns='Intercept')
def ps_fit(df):
    Xd = patsy.dmatrix(COV_FORMULA, df, return_type='dataframe')
    return sm.GLM(df['open'], Xd, family=sm.families.Binomial()).fit(disp=0).predict(Xd).values
def ow_est(df):
    ps = ps_fit(df); w = np.where(df['open']==1, 1-ps, ps); y = df['aki'].values; o = df['open'].values
    r1 = (w*y*o).sum()/(w*o).sum(); r0 = (w*y*(1-o)).sum()/(w*(1-o)).sum()
    return r1, r0, ps, w
r1, r0, ps, w = ow_est(d)
dd = d.reset_index(drop=True); groups = dd.groupby('subject_id').indices; keys = np.array(list(groups.keys())); rng = np.random.default_rng(20261028); bs=[]
for _ in range(500):
    idx = np.concatenate([groups[k] for k in rng.choice(keys, size=len(keys), replace=True)])
    try: a, b, _, _ = ow_est(dd.iloc[idx]); bs.append((a/b, a-b))
    except Exception: pass
bs = np.array(bs)
say(f"Overlap-weighted risks: open {r1:.1%}, endovascular {r0:.1%}; RR {r1/r0:.2f} ({np.percentile(bs[:,0],2.5):.2f} to {np.percentile(bs[:,0],97.5):.2f}); "
    f"RD {100*(r1-r0):.1f} points ({100*np.percentile(bs[:,1],2.5):.1f} to {100*np.percentile(bs[:,1],97.5):.1f}); 500 patient-clustered bootstrap reps")
RESULTS.append(dict(model='W1  Overlap-weighted risk ratio', N=len(d), events=int(d.aki.sum()), est=r1/r0, lo=np.percentile(bs[:,0],2.5), hi=np.percentile(bs[:,0],97.5), p=np.nan, measure='RR'))
o = d['open'].values
def wmean_var(col, wt):
    m = np.average(col, weights=wt); v = np.average((col-m)**2, weights=wt); return m, v
rows = []
for col in Xc.columns:
    x = Xc[col].values.astype(float)
    m1, v1 = wmean_var(x[o==1], np.ones((o==1).sum())); m0, v0 = wmean_var(x[o==0], np.ones((o==0).sum()))
    wm1, wv1 = wmean_var(x[o==1], w[o==1]); wm0, wv0 = wmean_var(x[o==0], w[o==0])
    sd = np.sqrt((v1+v0)/2) or 1; swd = np.sqrt((wv1+wv0)/2) or 1
    rows.append((col, (m1-m0)/sd, (wm1-wm0)/swd))
smd = pd.DataFrame(rows, columns=['covariate','SMD before','SMD after overlap weights']).round(3)
say("Standardized mean differences, open minus endovascular (|SMD| > 0.10 = imbalance by convention):"); say(smd.to_string(index=False))
say("Weighted effective sample size, open:", round(w[o==1].sum()**2/(w[o==1]**2).sum(),1), "| endovascular:", round(w[o==0].sum()**2/(w[o==0]**2).sum(),1))
say("Propensity score distribution (percentiles 1, 5, 50, 95, 99) open:", np.round(np.percentile(ps[o==1],[1,5,50,95,99]),3).tolist(), "| endovascular:", np.round(np.percentile(ps[o==0],[1,5,50,95,99]),3).tolist())
say("\n=== MISSINGNESS (count missing per model variable; small counts masked) ===")
mv = ['aki','approach','location','age_at_admission','gender','cci','anchor_year_group','rupture','urg','base24','vasopressor']
say(pd.Series({v: n_(d[v].isna().sum()) for v in mv}).to_string())

In [ ]:
# Cell 5: DESCRIPTIVE TABLE, unadjusted estimates, effect modification, mortality, S2b (verified vasopressors), R3 diagnostics (all post hoc)
say("\n=== CELL 5: DESCRIPTIVE AND DIAGNOSTIC ANALYSES (post hoc). '*' = also suppressed so a '<11' cell cannot be recovered by subtraction ===")
def cnt(x):   # same function as Cell 0 (redefined so this cell also works in an older open session)
    x = pd.DataFrame(x); out = x.astype(object).copy()
    for i in range(x.shape[0]):
        row = list(x.iloc[i]); small = [j for j, v in enumerate(row) if isinstance(v, (int, np.integer)) and 0 < v < SMALL_CELL]
        sec = []
        if len(small) == 1:
            oth = [(v, j) for j, v in enumerate(row) if j not in small and isinstance(v, (int, np.integer)) and v > 0]
            if oth: sec = [min(oth)[1]]
        for j in range(x.shape[1]):
            out.iat[i, j] = ('<%d' % SMALL_CELL) if j in small else ('*' if j in sec else row[j])
    return out

# ---- S2b: only drugs verified as vasopressors in the mimic-code concept files ----
V5 = [221289, 221662, 221749, 221906, 222315]      # epinephrine, dopamine, phenylephrine, norepinephrine, vasopressin (221653 = dobutamine is NOT included)
vv = []
for ch in pd.read_csv(F['inp'], usecols=['hadm_id','starttime','itemid'], parse_dates=['starttime'], chunksize=2_000_000):
    vv.append(ch[ch.itemid.isin(V5) & ch.hadm_id.isin(ids)])
d['vaso5'] = d.hadm_id.isin(set(pd.concat(vv).hadm_id)).astype(int)
say("Manuscript 'vasopressor' flag (any vasoactive: includes dobutamine and 4 unverified itemids) by verified-5-drug flag (rows = manuscript flag):")
say(cnt(pd.crosstab(d.vasopressor, d.vaso5)).to_string())
fit(d, 'aki', T(['vaso5']), 'S2b Add any verified vasopressor (5 drugs; post-procedure variable, mediator risk)', 'poisson')

# ---- unadjusted estimates (STROBE 16a) ----
say("\n--- Unadjusted estimates ---")
fit(d, 'aki', ["C(approach, Treatment('endovascular'))"], 'U1  UNADJUSTED modified Poisson RR (cluster by patient)', 'poisson')
d['open'] = (d.approach=='open').astype(int)
ols = smf.ols("aki ~ open", data=d).fit(cov_type='cluster', cov_kwds={'groups': d.subject_id.values})
say(f"U2  Unadjusted risk difference {100*ols.params['open']:.1f} points ({100*ols.conf_int().loc['open',0]:.1f} to {100*ols.conf_int().loc['open',1]:.1f}), cluster-robust")

# ---- Table 1 by approach (open vs endovascular) ----
say("\n--- Table 1 by approach (n or median [IQR]; percentages shown only for cells >= the small-cell minimum; SMD = open minus endovascular) ---")
g1 = d[d.approach=='open']; g0 = d[d.approach=='endovascular']
T1 = []
def med_iqr(s):
    q = s.quantile([.25,.5,.75]); return f"{q[.5]:.1f} ({q[.25]:.1f}-{q[.75]:.1f})"
def addc(name, col):
    a_, b_ = g1[col].dropna(), g0[col].dropna(); sd = np.sqrt((a_.var()+b_.var())/2) or 1
    T1.append((name, med_iqr(a_), med_iqr(b_), round((a_.mean()-b_.mean())/sd, 2), n_(d[col].isna().sum())))
def addcat(name, col):
    lv = [l for l in sorted(d[col].dropna().unique(), key=str)]
    cts = {g: [int((G[col]==l).sum()) for l in lv] for g, G in (('o', g1), ('e', g0))}
    hid = {}
    for g in cts:
        small = [i for i, k in enumerate(cts[g]) if 0 < k < SMALL_CELL]; sec = []
        if len(small) == 1:
            oth = [(k, i) for i, k in enumerate(cts[g]) if i not in small and k > 0]
            if oth: sec = [min(oth)[1]]
        hid[g] = set(small) | set(sec)
    for i, l in enumerate(lv):
        def f(g, G):
            k = cts[g][i]
            if i in hid[g]: return '<%d' % SMALL_CELL if 0 < k < SMALL_CELL else '*'
            return f"{k} ({100*k/len(G):.1f}%)"
        p1_, p0_ = cts['o'][i]/len(g1), cts['e'][i]/len(g0); sd = np.sqrt((p1_*(1-p1_)+p0_*(1-p0_))/2) or 1
        smd_ = '' if (i in hid['o'] or i in hid['e']) else round((p1_-p0_)/sd, 2)
        T1.append((f"{name}: {l}", f(('o'), g1), f('e', g0), smd_, ''))
T1.append(('N admissions', str(len(g1)), str(len(g0)), '', ''))
addc('Age, years', 'age_at_admission'); addc('Charlson index', 'cci'); addc('Baseline creatinine, mg/dL', 'base24'); addc('Length of stay, days', 'days')
for nm, cl in [('Sex', 'gender'), ('Aneurysm location', 'location'), ('Era', 'anchor_year_group'), ('Rupture', 'rupture'), ('Admission urgency', 'urg'), ('CKD stage group', 'ckd_group')]:
    addcat(nm, cl)
say(pd.DataFrame(T1, columns=['variable', f'open', 'endovascular', 'SMD', 'missing n']).to_string(index=False))

# ---- effect modification by aneurysm location ----
say("\n--- Aneurysm location (post hoc; within MIMIC-IV only; this is NOT the manuscript's MIMIC-versus-eICU ratio of 2.76) ---")
for L in ['abdominal', 'thoracic', 'thoracoabdominal']:
    sub = d[d.location==L]
    fit(sub, 'aki', [t for t in T() if 'location' not in t], f'L1 Stratum: {L} (N={n_(len(sub))})', 'poisson')
fi = "aki ~ C(approach, Treatment('endovascular'))*C(location, Treatment('abdominal')) + age_at_admission + C(gender) + cci + C(anchor_year_group) + rupture + C(urg, Treatment('elective'))"
try:
    m0_ = smf.glm(fi, data=d, family=sm.families.Poisson()).fit()
    mi = smf.glm(fi, data=d, family=sm.families.Poisson()).fit(cov_type='cluster', cov_kwds={'groups': d.loc[m0_.model.data.row_labels, 'subject_id'].values})
    its = [k for k in mi.params.index if ':' in k]
    for k in its:
        lo_, hi_ = np.exp(mi.conf_int().loc[k]); say(f"L2 Interaction ratio (RR in this location / RR in abdominal) {k.split(':')[-1]}: {np.exp(mi.params[k]):.2f} ({lo_:.2f} to {hi_:.2f}), p={mi.pvalues[k]:.3f}")
    R = np.zeros((len(its), len(mi.params))); [R.__setitem__((i, list(mi.params.index).index(k)), 1) for i, k in enumerate(its)]
    say(f"L3 Joint Wald test of approach x location: p={float(mi.wald_test(R, scalar=True).pvalue):.3f}")
except Exception as e:
    say("L2/L3 interaction model failed:", type(e).__name__, str(e)[:80])

# ---- in-hospital death as a secondary outcome ----
say("\n--- Secondary outcome ---")
fit(d, 'died', T(), 'M1  In-hospital death: modified Poisson RR, same covariates', 'poisson')

# ---- why does R3 (ICU-arrival cut) differ from K0/K/N? ----
say("\n--- R3 diagnostics ---")
if icu_path:
    d['icu_in'] = d.hadm_id.map(icu)
    d['in_R3'] = d.hadm_id.isin(d2.hadm_id).astype(int); d['in_K0'] = d.base_pre.notna().astype(int)
    ov = pd.crosstab(d.in_R3, d.in_K0)
    say("Overlap of the R3 subset (rows) with the pre-procedure-baseline subset K0 (columns: 0 = no, 1 = yes):")
    # both margins are known, so ONE hidden cell in a 2x2 table could be recovered: show the whole table or none
    say(ov.to_string() if not ((ov.values > 0) & (ov.values < SMALL_CELL)).any() else f"(table suppressed: at least one cell is below {SMALL_CELL}; the other diagnostics below still apply)")
    say("Approach composition: all 740 vs R3 subset vs K0 subset (counts):"); say(cnt(pd.DataFrame({'all': d.approach.value_counts(), 'R3': d[d.in_R3==1].approach.value_counts(), 'K0': d[d.in_K0==1].approach.value_counts()}).fillna(0).astype(int).T).to_string())
    say("Elective share: all", f"{(d.urg=='elective').mean():.1%}", "| R3 subset", f"{(d[d.in_R3==1].urg=='elective').mean():.1%}", "| K0 subset", f"{(d[d.in_K0==1].urg=='elective').mean():.1%}")
    say("Days from the procedure day to ICU arrival (25th, 50th, 75th), by approach (negative = ICU arrival before the procedure day):")
    say(((d.icu_in.dt.normalize()-d.proc_date).dt.days).groupby(d.approach).describe()[['25%','50%','75%']].to_string())
    say("Admissions with ANY creatinine charted before ICU arrival, by approach (share):"); say(d.assign(x=d.base_icu.notna()).groupby('approach').x.mean().round(3).to_string())
    fit(d[d.in_R3==1], 'aki', T(), 'R3-d1 Same R3 subset, PRIMARY outcome definition (baseline first 24 h, peak whole admission)', 'poisson')
    fit(d[(d.in_R3==1)&(d.in_K0==1)], 'aki', T(), 'R3-d3 In both R3 and K0 subsets, PRIMARY outcome definition', 'poisson')
else:
    say("R3 diagnostics skipped: icustays file not found.")

In [ ]:
# Cell 6: prints the full report (it contains everything from Cells 0-5, plus the results table)
out = pd.DataFrame(RESULTS)
out['Est (95% CI)'] = out.apply(lambda r: f"{r.est:.2f} ({r.lo:.2f} to {r.hi:.2f})", axis=1)
out['p'] = out.p.apply(lambda x: '' if pd.isna(x) else ('<0.001' if x<0.001 else f"{x:.3f}"))
out['events'] = out.events.apply(lambda v: n_(v))
print("=================== FULL REPORT (copy from here) ===================")
print("\n".join(REPORT))
print("\n=================== RESULTS TABLE ===================")
print(out[['model','N','events','measure','Est (95% CI)','p']].to_string(index=False))
print("=================== END ===================")

In [ ]:
# Cell 7 (add-on, run AFTER Run all in the same session): vasopressor definition after the item dictionary check
# d_items labels: 229617 Epinephrine; 229630/229631/229632 Phenylephrine (three formulations); 221653 Dobutamine (inotrope, not a vasopressor)
VP9 = [221289,221662,221749,221906,222315,229617,229630,229631,229632]
vv = []
for ch in pd.read_csv(F['inp'], usecols=['hadm_id','starttime','itemid'], chunksize=2_000_000):
    vv.append(ch[ch.itemid.isin(VP9+[221653]) & ch.hadm_id.isin(ids)])
vv = pd.concat(vv)
d['vp9'] = d.hadm_id.isin(vv[vv.itemid.isin(VP9)].hadm_id).astype(int)
d['dobu'] = d.hadm_id.isin(vv[vv.itemid==221653].hadm_id).astype(int)
say("\n=== CELL 7: vasopressor definition check ===")
say("Manuscript flag (any vasoactive) by 9-drug vasopressor flag (rows = manuscript flag); dobutamine-only admissions are the manuscript-flag=1, vp9=0 cell:")
say(cnt(pd.crosstab(d.vasopressor, d.vp9)).to_string())
say("Dobutamine exposure (any):", n_(d.dobu.sum()))
fit(d, 'aki', T(['vp9']), 'S2c Add any of 9 verified vasopressors (epinephrine, norepinephrine, phenylephrine incl. new formulations, dopamine, vasopressin; excludes dobutamine)', 'poisson')


In [ ]:
# Cell 8: MIMIC-IV thoracic vs abdominal effect, same covariates the eICU model can use (eICU has no approach and no thoracoabdominal class)
say("\n=== CELL 8: location effect in MIMIC-IV (for the MIMIC-eICU comparison; post hoc) ===")
dl = d[d.location.isin(['abdominal','thoracic'])].copy()
for lab, extra in [('8a  no approach (comparable with eICU)', ''), ('8b  with approach', " + C(approach, Treatment('endovascular'))")]:
    f = "aki ~ C(location, Treatment('abdominal')) + age_at_admission + C(gender) + cci + rupture" + extra
    try:
        m0 = smf.glm(f, data=dl, family=sm.families.Poisson()).fit()
        m = smf.glm(f, data=dl, family=sm.families.Poisson()).fit(cov_type='cluster', cov_kwds={'groups': dl.loc[m0.model.data.row_labels, 'subject_id'].values})
        k = [x for x in m.params.index if 'thoracic' in x][0]; b, se = m.params[k], m.bse[k]
        say(f"{lab}: N={int(m.nobs)}, events={n_(dl.aki.sum())}, thoracic vs abdominal RR {np.exp(b):.2f} ({np.exp(b-1.96*se):.2f} to {np.exp(b+1.96*se):.2f}) | log RR {b:.4f}, SE {se:.4f}")
    except Exception as e: say(f"{lab}: MODEL FAILED ({type(e).__name__}: {str(e)[:70]})")
say("(MIMIC covariate 'cci' is ICD-based; the eICU count is text-derived. The two are not the same measure.)")
print("\n".join(REPORT[-4:]))


In [ ]:
# Cell 9 v2.4 (add-on, run AFTER Run all in the same session): renal replacement therapy (RRT, dialysis) during the stay. POST HOC, added 2 Nov 2026.
# v2.4 changes PRINTING ONLY (no estimate changes): removed the printouts that allowed small counts to be recovered by subtraction.
# Why: this outcome does not depend on any baseline creatinine, so it speaks to the baseline-timing weakness and to the null stage 3 result.
# It finds RRT items in the item dictionary by label, prints exactly which items it used, and also uses RRT procedure codes.
import re
say("\n=== CELL 9: renal replacement therapy (dialysis) during the stay (post hoc) ===")
DI_CAND = [P+"MIMIC 4/d_items.csv.gz", P+"mimic-iv/icu/d_items.csv.gz", P+"mimic-iv/d_items.csv.gz"]
PE_CAND = [P+"MIMIC 4/procedureevents.csv.gz", P+"mimic-iv/icu/procedureevents.csv.gz", P+"mimic-iv/procedureevents.csv.gz"]
di_path = next((p for p in DI_CAND if os.path.exists(p)), None); pe_path = next((p for p in PE_CAND if os.path.exists(p)), None)
if not di_path or not pe_path:
    raise SystemExit("CELL 9 needs d_items and procedureevents. Missing: " + ", ".join(n for n, p in [('d_items', di_path), ('procedureevents', pe_path)] if not p)
                     + ". Edit DI_CAND / PE_CAND above to the right path (procedureevents is in the MIMIC-IV icu folder).")
di = pd.read_csv(di_path, usecols=['itemid', 'label', 'linksto'])
lab = di.label.fillna('').str.lower()
hit = di[lab.str.contains(r'dialysis|crrt|cvvh|scuf|hemofiltration|ultrafiltration|hemodialysis', regex=True)].copy()
hit['is_access'] = hit.label.fillna('').str.lower().str.contains(r'catheter|access|cannula|line |circuit|filter change|priming', regex=True)
pe = pd.read_csv(pe_path, usecols=['hadm_id', 'starttime', 'itemid'], parse_dates=['starttime'])
pe = pe[pe.hadm_id.isin(ids) & pe.itemid.isin(hit.itemid)]
hit['cohort_admissions'] = hit.itemid.map(pe.groupby('itemid').hadm_id.nunique()).fillna(0).astype(int)
show = hit.assign(cohort_admissions=hit.cohort_admissions.map(n_), used=np.where(hit.is_access, 'NO (access/catheter/circuit item)', 'YES'))[['itemid', 'label', 'linksto', 'cohort_admissions', 'used']]
say("Dictionary items matching dialysis / CRRT / hemofiltration (counts = cohort admissions, small cells masked):"); say(show.to_string(index=False))
RRT_ITEMS = hit[~hit.is_access].itemid.tolist()
pe_use = pe[pe.itemid.isin(RRT_ITEMS)].merge(d[['hadm_id', 'proc_date']], on='hadm_id')
# ICD procedure codes (already loaded in Cell 0 as pr): ICD-9 39.95 hemodialysis, 54.98 peritoneal dialysis; ICD-10-PCS 5A1D* urinary filtration
ic = pr[((pr.icd_version == 9) & pr.icd_code.isin(['3995', '5498'])) | ((pr.icd_version == 10) & pr.icd_code.str.startswith('5A1D'))].merge(d[['hadm_id', 'proc_date']], on='hadm_id')
d['rrt_item_any'] = d.hadm_id.isin(pe_use.hadm_id).astype(int)
d['rrt_item_post'] = d.hadm_id.isin(pe_use[pe_use.starttime.dt.normalize() >= pe_use.proc_date].hadm_id).astype(int)
d['rrt_icd_any'] = d.hadm_id.isin(ic.hadm_id).astype(int)
d['rrt_icd_post'] = d.hadm_id.isin(ic[ic.chartdate >= ic.proc_date].hadm_id).astype(int)
d['rrt_any'] = ((d.rrt_item_any == 1) | (d.rrt_icd_any == 1)).astype(int)
d['rrt_post'] = ((d.rrt_item_post == 1) | (d.rrt_icd_post == 1)).astype(int)
# Only the two totals are printed. The item-only / code-only / both split and the RRT-by-AKI table are deliberately NOT printed (v2.4):
# together with the totals they let a reader recover a count below 11 by subtraction.
say("RRT any time in the admission:", n_(d.rrt_any.sum()), "| on or after the procedure day (the outcome analysed):", n_(d.rrt_post.sum()))
say("RRT on/after the procedure day by approach (rows; columns 0 = no RRT, 1 = RRT; small cells masked):"); say(cnt(pd.crosstab(d.approach, d.rrt_post)).to_string())
say("Crude RRT risk: open", f"{d[d.approach=='open'].rrt_post.mean():.1%}", "| endovascular", f"{d[d.approach=='endovascular'].rrt_post.mean():.1%}")
say("RRT by KDIGO stage 3 flag (rows = ratio>=3.0: 0/1):"); say(cnt(pd.crosstab(d.stage3, d.rrt_post)).to_string())
ev = int(d.rrt_post.sum())
fit(d, 'rrt_post', [T()[0]], 'D0  UNADJUSTED: RRT on/after the procedure day', 'poisson')
if ev >= 20:
    fit(d, 'rrt_post', T(), 'D1  ADJUSTED (primary covariates): RRT on/after the procedure day', 'poisson')
    fit(d, 'rrt_any', T(), 'D2  ADJUSTED: any RRT during the admission', 'poisson')
    d['rrt_or_death'] = ((d.rrt_post == 1) | (d.died == 1)).astype(int)
    fit(d, 'rrt_or_death', T(), 'D3  ADJUSTED: RRT or in-hospital death', 'poisson')
else:
    say(f"Only {n_(ev)} RRT events: too few for a stable adjusted model, so only the counts and the unadjusted estimate are reported.")
say("(ESRD / chronic dialysis admissions were excluded from the cohort by the diagnosis-code rule, so RRT here is mostly new; a patient with undetected chronic dialysis would still be counted.)")


In [ ]:
# Cell 10 (add-on, run AFTER Run all in the same session): TIME-ORDERED AKI. POST HOC, added 2 Nov 2026.
# Why: in the primary definition the peak is the highest creatinine at ANY time up to discharge (including before admission) and baseline is the lowest
# value in the first 24 h; nothing requires the peak to come AFTER the baseline value. A high early value that then falls (recovery) could be counted as AKI.
# Here the peak must be charted strictly AFTER the baseline value. Aggregates only; small counts are masked or withheld.
say("\n=== CELL 10: time-ordered AKI (peak must be charted after the baseline value) (post hoc) ===")
cc = cr[['hadm_id', 'charttime', 'valuenum']].copy()
cc = cc.merge(d[['hadm_id', 'admittime']], on='hadm_id')
f24_ = cc[cc.charttime <= cc.admittime + pd.Timedelta('24h')].sort_values(['hadm_id', 'charttime'])
tb = f24_.loc[f24_.groupby('hadm_id').valuenum.idxmin(), ['hadm_id', 'charttime', 'valuenum']].rename(columns={'charttime': 't_base', 'valuenum': 'base_chk'})
chk = d[['hadm_id', 'base24']].merge(tb, on='hadm_id', how='left')
say("CHECK: baseline value/time recomputed; disagreements with the baseline used so far:", n_(((chk.base24 - chk.base_chk).abs() > 1e-9).sum()), "(expect 0; the one admission with no baseline is excluded from this check)" )
post = cc.merge(tb[['hadm_id', 't_base']], on='hadm_id'); post = post[post.charttime > post.t_base]
pk_after = post.groupby('hadm_id').valuenum.max().rename('peak_after')
allmax = cc.loc[cc.groupby('hadm_id').valuenum.idxmax(), ['hadm_id', 'charttime']].rename(columns={'charttime': 't_peak'})
d = d.drop(columns=[x for x in ['peak_after', 't_base', 't_peak', 'aki_to', 'stage3_to', 'peak_before_base'] if x in d.columns]) \
     .merge(pk_after, on='hadm_id', how='left').merge(tb[['hadm_id', 't_base']], on='hadm_id', how='left').merge(allmax, on='hadm_id', how='left')
d['aki_to'] = ((d.peak_after / d.base24) >= 1.5).astype(int)
d['stage3_to'] = ((d.peak_after / d.base24) >= 3.0).astype(int)
d['peak_before_base'] = (d.t_peak <= d.t_base).astype(int)
say("Admissions whose highest creatinine was charted at or before the baseline value:", n_(d.peak_before_base.sum()), "of", int(d.base24.notna().sum()),
    "| among those flagged AKI by the primary definition:", n_(((d.peak_before_base == 1) & (d.aki == 1)).sum()))
say("By approach (rows; columns 0/1 = highest value charted at or before baseline; small cells masked):"); say(cnt(pd.crosstab(d.approach, d.peak_before_base)).to_string())
disc = int(((d.aki == 1) & (d.aki_to == 0)).sum()); disc3 = int(((d.stage3 == 1) & (d.stage3_to == 0)).sum())
say("Admissions classified AKI by the primary definition but NOT by the time-ordered definition:", n_(disc))
withhold = 0 < disc < SMALL_CELL; withhold3 = 0 < disc3 < SMALL_CELL
if withhold: say("(Event totals for the time-ordered AKI models are withheld below: the total would reveal a count under", SMALL_CELL, "by subtraction from 229. Report N and the risk ratio only.)")
def show_to(y, covs, label, wh):
    m = fit(d, y, covs, label, 'poisson', quiet=True)
    if m is None: return
    r = RESULTS[-1]
    say(f"{label}: N={r['N']}, events={'withheld' if wh else n_(r['events'])}, open vs endovascular RR {r['est']:.2f} ({r['lo']:.2f} to {r['hi']:.2f}), p={r['p']:.3f}")
show_to('aki_to', [T()[0]], 'TO0  UNADJUSTED time-ordered AKI', withhold)
show_to('aki_to', T(), 'TO1  ADJUSTED (primary covariates) time-ordered AKI', withhold)
show_to('stage3_to', T(), 'TO2  ADJUSTED time-ordered stage 3 (ratio >= 3.0)', withhold3)
say("Reference, same covariates, primary definition: RR 1.87 (1.49 to 2.34), 229 events. Reference, absolute-rise criterion (already time-ordered, forward 48 h window): RR 1.65 (1.41 to 1.94).")


In [ ]:
# Cell 11 (add-on, run AFTER Run all in the same session): CONCOMITANT CARDIAC / ASCENDING-ARCH PROCEDURE CODES. POST HOC, added 2 Nov 2026.
# Why: the cohort was defined by descending/abdominal aortic repair codes; admissions with ascending-aorta, arch, aortic valve or coronary bypass codes were kept.
# They are commoner in thoracic repairs and could raise AKI risk on their own. Codes are found from the procedure DICTIONARY by title (printed below so they can be checked).
# Aggregates only.
say("\n=== CELL 11: concomitant ascending/arch, aortic valve, coronary bypass and bypass-circuit codes (post hoc) ===")
DP_CAND = [P+"mimic-iv/hosp/d_icd_procedures.csv.gz", P+"MIMIC 4/d_icd_procedures.csv.gz", P+"mimic-iv/d_icd_procedures.csv.gz"]
dp_path = next((p for p in DP_CAND if os.path.exists(p)), None)
if not dp_path: raise SystemExit("CELL 11 needs d_icd_procedures.csv.gz (MIMIC-IV hosp folder). Edit DP_CAND above to its path.")
dp = pd.read_csv(dp_path, dtype={'icd_code': str}); dp['t'] = dp.long_title.fillna('').str.lower()
GROUPS = {'valve': r'aortic valve',
          'asc_arch': r'thoracic aorta, ascending|ascending aorta|aortic arch',
          'cabg': r'coronary artery bypass|aortocoronary|bypass coronary|coronary bypass',
          'cpb': r'extracorporeal circulation|cardiopulmonary bypass'}
pr_ = pr[['hadm_id', 'icd_code', 'icd_version']].copy(); pr_['icd_code'] = pr_.icd_code.astype(str)
flag = {}
for g, pat in GROUPS.items():
    m = dp[dp.t.str.contains(pat, regex=True)][['icd_code', 'icd_version', 'long_title']]
    hit_ = pr_.merge(m, on=['icd_code', 'icd_version'])
    flag[g] = set(hit_.hadm_id)
    per = hit_.groupby(['icd_version', 'icd_code', 'long_title']).hadm_id.nunique().reset_index(name='n')
    per['n'] = per.n.map(n_)
    say(f"\nGroup '{g}': dictionary rows matched {len(m)}; codes present in the cohort (admissions, small counts masked):")
    say(per.sort_values(['icd_version', 'icd_code']).to_string(index=False) if len(per) else "  (none)")
    d[g] = d.hadm_id.isin(flag[g]).astype(int)
d['card_any'] = ((d.valve == 1) | (d.asc_arch == 1) | (d.cabg == 1)).astype(int)
say("\nAdmissions with an ascending/arch, aortic valve or coronary bypass code (card_any):", n_(d.card_any.sum()), "of", len(d),
    "| bypass-circuit code (cpb):", n_(d.cpb.sum()))
say("card_any by aneurysm location (rows; columns 0/1; small cells masked):"); say(cnt(pd.crosstab(d.location, d.card_any)).to_string())
say("card_any by approach (rows; columns 0/1; small cells masked):"); say(cnt(pd.crosstab(d.approach, d.card_any)).to_string())
k = int(d[d.card_any == 1].aki.sum()); wh = 0 < k < SMALL_CELL or 0 < (int(d.card_any.sum()) - k) < SMALL_CELL
say("AKI among card_any admissions:", n_(k), "| risk", f"{d[d.card_any==1].aki.mean():.1%}" if d.card_any.sum() > 0 else "NA", "| AKI risk without:", f"{d[d.card_any==0].aki.mean():.1%}")
if wh: say("(Event totals below are withheld where they would reveal a count under", SMALL_CELL, "by subtraction.)")
def show11(data, covs, label, hide):
    m = fit(data, 'aki', covs, label, 'poisson', quiet=True)
    if m is None: return
    r = RESULTS[-1]
    say(f"{label}: N={r['N']}, events={'withheld' if hide else n_(r['events'])}, open vs endovascular RR {r['est']:.2f} ({r['lo']:.2f} to {r['hi']:.2f}), p={r['p']:.3f}")
show11(d[d.card_any == 0], T(), 'CC1  EXCLUDING admissions with these codes (primary covariates)', wh)
show11(d, T(['card_any']), 'CC2  ADJUSTED for card_any (primary covariates + indicator)', False)
show11(d, T(['card_any', 'cpb']), 'CC3  ADJUSTED for card_any and bypass-circuit code', False)
show11(d[(d.card_any == 0) & (d.location != 'thoracic')], T(), 'CC4  Excluding these codes AND thoracic repairs (abdominal + thoracoabdominal only)', wh)
say("Reference, primary: RR 1.87 (1.49 to 2.34), 229 events. Earlier logistic-model checks (not comparable): exclusion OR 1.97 (1.29 to 3.01); adjusted OR 1.67 (1.09 to 2.56).")
